# Day 28 — Practice SOLUTIONS
### Hypothesis Testing · Module 2 · Topic 2.8

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it tests | Count |
|---|---|---|
| **A** | Predict the output | 8 |
| **B** | Spot & fix the bug | 4 |
| **C** | Write the code | 5 |
| **D** | Challenge | 2 |

**How to work:** for Part A, write your answer down *before* running anything.

Three questions to keep asking yourself:

- Is this **paired** or two independent groups?
- **How many tests** have I actually run?
- Am I reporting the **effect**, or just the verdict?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.simplefilter("ignore")

rng = np.random.default_rng(28)
print("ready")

---
# Part A — Predict the output

Write your answer first. Then run the cell.

### A1

```python
a = rng.normal(0, 1, (20_000, 25))
b = rng.normal(0, 1, (20_000, 25))
p = stats.ttest_ind(a, b, axis=1).pvalue
print(round((p < 0.05).mean(), 3))
print(round(np.median(p), 3))
```

There is no difference between the groups at all. What do these two print?

In [ ]:
a = rng.normal(0, 1, (20_000, 25))
b = rng.normal(0, 1, (20_000, 25))
p = stats.ttest_ind(a, b, axis=1).pvalue
print(round((p < 0.05).mean(), 3))
print(round(np.median(p), 3))

**About `0.05` and about `0.50`.**

With no effect at all, the test still fires 5% of the time — that is what α = 0.05 *means*, not
a defect.

The median p-value is 0.5 because **under the null, p-values are uniform on [0,1]**. Every
value is equally likely: 0.03 is exactly as likely as 0.97.

That uniformity is the cleanest definition of a p-value there is, and most people have never
seen it.

In [ ]:
print(f"fires {(p<0.05).mean():.4f} of the time   (alpha = 0.05)")
print(f"median p = {np.median(p):.4f}             (uniform -> 0.5)")
print(f"KS test against uniform: p = {stats.kstest(p,'uniform').pvalue:.3f}")
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.hist(p, bins=25, color="#9AA7B8"); ax.axhline(20_000/25, color="#152238", ls="--", lw=2)
ax.set_xlabel("p-value"); ax.set_title("Flat. Every p-value equally likely.")
plt.show()

### A2

```python
a = rng.normal(0, 5, 10)      # small group, big spread
b = rng.normal(0, 1, 50)      # large group, small spread
print(round(stats.ttest_ind(a, b).pvalue, 4))
print(round(stats.ttest_ind(a, b, equal_var=False).pvalue, 4))
```

Both groups come from a distribution with mean 0 — there is no real difference. Will the two
p-values agree?

In [ ]:
a = rng.normal(0, 5, 10)
b = rng.normal(0, 1, 50)
print(round(stats.ttest_ind(a, b).pvalue, 4))
print(round(stats.ttest_ind(a, b, equal_var=False).pvalue, 4))

**No — and they can disagree dramatically.**

The first is Student's t, which assumes both groups have the same variance. They do not: the
*smaller* group has the *larger* spread, which is the configuration Student's t handles worst.

Repeat this setup thousands of times with no real effect and Student's t fires about **36%** of
the time instead of 5%. Welch holds at 5%.

`equal_var=False` is the fix, and `equal_var=True` is scipy's default — so the dangerous
version is the one you get by typing the obvious thing.

In [ ]:
r = np.random.default_rng(7)
T = 20_000
A_ = r.normal(0, 5, (T, 10)); B_ = r.normal(0, 1, (T, 50))
s_ = (stats.ttest_ind(A_, B_, axis=1).pvalue < 0.05).mean()
w_ = (stats.ttest_ind(A_, B_, axis=1, equal_var=False).pvalue < 0.05).mean()
print(f"over {T:,} repeats with NO real difference:")
print(f"  Student's t fires {s_:.1%} of the time   <- should be 5%")
print(f"  Welch's   t fires {w_:.1%} of the time")
print()
import inspect
print("scipy default for equal_var:",
      inspect.signature(stats.ttest_ind).parameters['equal_var'].default)

### A3

```python
print(round(1 - 0.95 ** 20, 3))
print(round(0.05 / 20, 4))
```

Two numbers about the same situation. What are they, and how do they relate?

In [ ]:
print(round(1 - 0.95 ** 20, 3))
print(round(0.05 / 20, 4))

**`0.642` and `0.0025`.**

The first is the chance of getting **at least one** false positive when you run 20 tests on
data with nothing in it. Roughly two times in three, pure noise will hand you a "discovery".

The second is the Bonferroni correction: to keep the *overall* false positive rate at 5% across
20 tests, each individual test needs a threshold of 0.05/20 = 0.0025.

They are two sides of the same arithmetic. A 20-column correlation matrix, by the way, is
**190** tests.

In [ ]:
print(f"{'tests':>7} {'P(>=1 false positive)':>24} {'Bonferroni threshold':>22}")
for k in [1, 5, 10, 20, 50, 190]:
    print(f"{k:>7} {1-0.95**k:>23.1%} {0.05/k:>22.5f}")
print()
print("190 is the number of pairs in a 20-column correlation matrix:")
print(f"  20 choose 2 = {20*19//2}")

### A4

```python
person = rng.normal(100, 20, 20)
before = person + rng.normal(0, 3, 20)
after  = person + rng.normal(0, 3, 20) + 4
print(round(stats.ttest_ind(before, after).pvalue, 4))
print(round(stats.ttest_rel(after, before).pvalue, 6))
```

There is a real +4 effect. Which test finds it?

In [ ]:
person = rng.normal(100, 20, 20)
before = person + rng.normal(0, 3, 20)
after  = person + rng.normal(0, 3, 20) + 4
print(round(stats.ttest_ind(before, after).pvalue, 4))
print(round(stats.ttest_rel(after, before).pvalue, 6))

**Only the paired test — and the gap is enormous.**

The unpaired test returns something unremarkable; the paired test returns a p-value several
orders of magnitude smaller.

**Why:** people differ from each other by about 20 points, and the effect is 4. The unpaired
test has to see a 4-point signal through 20 points of between-person noise. The paired test
subtracts each person from themselves, so that noise cancels completely and only the change
remains.

Repeated across many simulations, the unpaired test detects this effect about **0%** of the
time and the paired test about **98%**. Same data, same effect.

In [ ]:
r = np.random.default_rng(285)
T, n = 4000, 20
pr = r.normal(100, 20, (T, n))
bef = pr + r.normal(0, 3, (T, n))
aft = pr + r.normal(0, 3, (T, n)) + 4
print(f"unpaired detects it {(stats.ttest_ind(bef,aft,axis=1).pvalue<0.05).mean():>6.1%}")
print(f"paired   detects it {(stats.ttest_rel(bef,aft,axis=1).pvalue<0.05).mean():>6.1%}")
print()
print(f"sd between people   {pr.std():.1f}")
print(f"sd of the change    {(aft-bef).std():.1f}   <- what pairing leaves behind")

### A5

```python
for n in [30, 1000, 100_000]:
    p = 2 * (1 - stats.norm.cdf(0.02 * np.sqrt(n/2)))
    print(n, round(p, 6))
```

The true effect is d = 0.02 in every row — a difference so small it is meaningless. What
happens to the p-value?

In [ ]:
for n in [30, 1000, 100_000]:
    p = 2 * (1 - stats.norm.cdf(0.02 * np.sqrt(n/2)))
    print(n, round(p, 6))

**It collapses: roughly `0.94`, `0.66`, `0.000008`.**

The effect never changes. Only n does.

No real-world difference is *exactly* zero, so with enough data **everything** becomes
statistically significant. A company with ten million users will find every button colour
change significant.

"Significant" means **detectable**, not **important**. The number that answers "is this worth
caring about?" is the effect size, and it does not move at all across those three rows.

In [ ]:
print(f"{'n/group':>9} {'p-value':>12} {'Cohen d':>10} {'verdict':>18}")
for n in [30, 100, 1000, 10_000, 100_000, 1_000_000]:
    p = 2*(1 - stats.norm.cdf(0.02*np.sqrt(n/2)))
    print(f"{n:>9} {p:>12.2e} {0.02:>10.2f} "
          f"{'significant' if p < 0.05 else 'not significant':>18}")

### A6

```python
def ppv(prior, power=0.80, alpha=0.05):
    return prior*power / (prior*power + (1-prior)*alpha)

print(round(ppv(0.50), 3))
print(round(ppv(0.05), 3))
```

Both describe results that came back with p < 0.05. What are these numbers?

In [ ]:
def ppv(prior, power=0.80, alpha=0.05):
    return prior*power / (prior*power + (1-prior)*alpha)

print(round(ppv(0.50), 3))
print(round(ppv(0.05), 3))

**`0.941` and `0.457`.**

This is the fraction of your "significant" results that are actually real.

- If **half** the hypotheses you test are true, 94% of your significant findings are real.
- If only **5%** are true — which is optimistic for exploratory work — then **fewer than half**
  are.

Identical p-values, identical methods, opposite trustworthiness. The difference is entirely in
*what you were testing in the first place*, and the p-value cannot see that.

This is why "p < 0.05" does not mean "95% sure", and it is the clearest arithmetic explanation
of the replication crisis.

In [ ]:
print(f"{'% hypotheses true':>19} {'80% power':>11} {'50% power':>11} {'30% power':>11}")
for prior in [0.50, 0.20, 0.10, 0.05, 0.01]:
    print(f"{prior:>18.0%} {ppv(prior,0.8):>11.1%} {ppv(prior,0.5):>11.1%} {ppv(prior,0.3):>11.1%}")
print()
print("Worked through with 1,000 hypotheses, 100 true, 80% power:")
print(f"   true findings  {100*0.80:>5.0f}")
print(f"   false alarms   {900*0.05:>5.0f}")
print(f"   so {45/125:.0%} of 'discoveries' are wrong")

### A7

```python
d = rng.normal(0, 1, 8)
se = d.std(ddof=1) / np.sqrt(8)
print(round(1.96 * se, 4))
print(round(stats.t.ppf(0.975, 7) * se, 4))
```

Which of these gives a genuine 95% interval at n = 8?

In [ ]:
d = rng.normal(0, 1, 8)
se = d.std(ddof=1) / np.sqrt(8)
print(round(1.96 * se, 4))
print(round(stats.t.ppf(0.975, 7) * se, 4))

**The second. The t critical value at 7 degrees of freedom is 2.365, not 1.96** — about 21%
wider.

When you use 1.96 you are pretending you know σ. You do not: you estimated it from the same
eight observations, and that estimate is itself uncertain. The t distribution has heavier tails
to pay for that extra uncertainty.

Measured coverage at n = 8: using 1.96 gives about **91%**, using t gives **95%**. By n = 100
they agree to the third decimal, which is why nobody worries about it in large studies.

In [ ]:
print(f"{'n':>5} {'t critical':>12} {'z critical':>12} {'t coverage':>12} {'z coverage':>12}")
r = np.random.default_rng(284)
for n in [3, 5, 8, 15, 30, 100]:
    s = r.normal(0, 1, (40_000, n))
    se_ = s.std(ddof=1, axis=1)/np.sqrt(n)
    tc = stats.t.ppf(.975, n-1)
    print(f"{n:>5} {tc:>12.3f} {1.96:>12.3f} "
          f"{(np.abs(s.mean(axis=1))<=tc*se_).mean():>11.1%} "
          f"{(np.abs(s.mean(axis=1))<=1.96*se_).mean():>11.1%}")

### A8

```python
x = rng.normal(0, 1, 60)
r1 = stats.ttest_1samp(x, 0)
r2 = stats.ttest_1samp(x + 10, 10)
print(round(r1.pvalue, 4) == round(r2.pvalue, 4))
print(round(r1.statistic, 4) == round(r2.statistic, 4))
```

What prints, and what does it tell you about what a test actually measures?

In [ ]:
x = rng.normal(0, 1, 60)
r1 = stats.ttest_1samp(x, 0)
r2 = stats.ttest_1samp(x + 10, 10)
print(round(r1.pvalue, 4) == round(r2.pvalue, 4))
print(round(r1.statistic, 4) == round(r2.statistic, 4))

**`True` and `True`.**

Shifting every value by 10 and shifting the null by 10 changes nothing. The test only ever
looks at the **gap between what you saw and what you expected**, measured in standard errors.

That is the whole formula: `t = (observed − expected) / SE`. The absolute scale is invisible to
it.

Which is also a warning: a test statistic of 3.35 tells you nothing about whether the effect is
*big*. It is three and a bit standard errors — and a standard error can be 0.01 or 10,000.

In [ ]:
print(f"original: mean {x.mean():+.4f}  t = {r1.statistic:+.4f}  p = {r1.pvalue:.4f}")
print(f"shifted : mean {(x+10).mean():+.4f}  t = {r2.statistic:+.4f}  p = {r2.pvalue:.4f}")
print()
print("Same t, same p. Only the GAP matters, measured in standard errors.")
print()
print("which is why the effect size has to be reported separately:")
print(f"  Cohen's d = {x.mean()/x.std(ddof=1):+.4f}   (unchanged by the shift too)")

---
# Part B — Spot & fix the bug

Each cell runs without error and gives a **wrong** or **badly reported** answer. Find the
problem, say what it means, and write the fix.

### B1 — the default

A team compares a small pilot group against the existing user base.

In [ ]:
rng_b1 = np.random.default_rng(404)
pilot    = rng_b1.normal(50, 18, 12)      # 12 people, highly variable
existing = rng_b1.normal(50, 4, 200)      # 200 people, consistent

res = stats.ttest_ind(pilot, existing)
print(f"p = {res.pvalue:.4f}  ->  the pilot group is significantly different!")

**Bug:** `ttest_ind` defaults to `equal_var=True` — Student's t, which assumes the two groups
have the same variance. Here the *smaller* group has a spread four times larger, which is the
exact configuration Student's t handles worst.

**What the wrong number means:** nothing. Both groups were drawn with a mean of 50 — there is
no difference at all. Simulate this exact setup repeatedly and Student's t declares a difference
around **52%** of the time — it is wrong more often than it is right.

**Fix:** `stats.ttest_ind(pilot, existing, equal_var=False)` — Welch's t, which holds at 5%.

The uncomfortable part is that nothing in the output hints at the problem. You get a p-value
either way.

In [ ]:
rng_b1 = np.random.default_rng(404)
pilot    = rng_b1.normal(50, 18, 12)
existing = rng_b1.normal(50, 4, 200)

print(f"pilot:    n={len(pilot):>3}  mean {pilot.mean():6.2f}  sd {pilot.std(ddof=1):6.2f}")
print(f"existing: n={len(existing):>3}  mean {existing.mean():6.2f}  sd {existing.std(ddof=1):6.2f}")
print()
print(f"Student (default): p = {stats.ttest_ind(pilot, existing).pvalue:.4f}")
print(f"Welch            : p = {stats.ttest_ind(pilot, existing, equal_var=False).pvalue:.4f}")
print()
#  how often is Student wrong in this setup?
r = np.random.default_rng(1)
T = 20_000
A_ = r.normal(50, 18, (T, 12)); B_ = r.normal(50, 4, (T, 200))
print(f"over {T:,} repeats with NO real difference:")
print(f"  Student fires {(stats.ttest_ind(A_,B_,axis=1).pvalue<0.05).mean():.1%}")
print(f"  Welch   fires {(stats.ttest_ind(A_,B_,axis=1,equal_var=False).pvalue<0.05).mean():.1%}")

### B2 — the wrong test

A clinic measures each patient's blood pressure before and after a six-week programme.

In [ ]:
rng_b2 = np.random.default_rng(402)
patient = rng_b2.normal(140, 18, 30)            # patients differ a lot
before  = patient + rng_b2.normal(0, 4, 30)
after   = patient + rng_b2.normal(0, 4, 30) - 6  # the programme really works

res = stats.ttest_ind(before, after)
print(f"p = {res.pvalue:.4f}  ->  no significant improvement. The programme failed.")

**Bug:** the same 30 patients are measured twice, so the data is **paired** — but
`ttest_ind` treats the two columns as independent groups.

**What the wrong number means:** the unpaired test is trying to detect a 6-point improvement
through 18 points of between-patient variation. It cannot, and it reports a null result on a
programme that genuinely works.

**Fix:** `stats.ttest_rel(after, before)`, which subtracts each patient from themselves. The
between-patient variation cancels entirely and the p-value drops by orders of magnitude.

**The cost of this bug is the opposite of B1's:** B1 invented an effect that was not there,
this one discards an effect that was. Both come from one default and one function name.

In [ ]:
rng_b2 = np.random.default_rng(402)
patient = rng_b2.normal(140, 18, 30)
before  = patient + rng_b2.normal(0, 4, 30)
after   = patient + rng_b2.normal(0, 4, 30) - 6

print(f"unpaired: p = {stats.ttest_ind(before, after).pvalue:.4f}")
print(f"paired  : p = {stats.ttest_rel(after, before).pvalue:.2e}")
print()
change = after - before
lo, hi = stats.t.interval(0.95, 29, change.mean(), change.std(ddof=1)/np.sqrt(30))
print(f"mean change {change.mean():+.2f} mmHg   95% CI [{lo:+.2f}, {hi:+.2f}]")
print()
print(f"sd between patients {patient.std(ddof=1):6.2f}   <- what the unpaired test fights")
print(f"sd of the change    {change.std(ddof=1):6.2f}   <- what the paired test uses")

fig, ax = plt.subplots(figsize=(5.5, 3))
for i in range(30):
    ax.plot([0, 1], [before[i], after[i]], color="#9AA7B8", lw=1, marker="o", ms=3)
ax.set_xticks([0, 1]); ax.set_xticklabels(["before", "after"])
ax.set_ylabel("blood pressure"); ax.set_title("Almost every patient improves")
plt.show()

### B3 — the report

An analyst reports the result of an A/B test on 80,000 users per arm.

In [ ]:
rng_b3 = np.random.default_rng(403)
control = rng_b3.normal(4.50, 3.0, 80_000)       # seconds on page
variant = rng_b3.normal(4.53, 3.0, 80_000)       # a 0.03 second difference

res = stats.ttest_ind(control, variant, equal_var=False)
print(f"p = {res.pvalue:.2e}")
print("The new design significantly increases time on page. Ship it.")

**Bug:** not in the test — it is correct, and Welch was used properly. The bug is in the
**conclusion**.

**What the number means:** with 80,000 users per arm the test can detect a difference of **0.03
seconds**. That is real, and it is also completely meaningless as a product decision.

**Fix:** report the effect size and the confidence interval, and decide against a threshold you
set *before* the test:

- the difference is about 0.03 seconds, 95% CI roughly 0.004 to 0.06
- Cohen's d is about 0.01, which is a hundredth of a "small" effect

**The judgement, not the statistic:** "0.03 seconds is below anything we would act on, so this
change is not worth shipping — and we now know that precisely." That is a *more* useful
conclusion than the significance test, and it comes from the interval.

In [ ]:
rng_b3 = np.random.default_rng(403)
control = rng_b3.normal(4.50, 3.0, 80_000)
variant = rng_b3.normal(4.53, 3.0, 80_000)

diff = variant.mean() - control.mean()
se = np.sqrt(control.var(ddof=1)/80_000 + variant.var(ddof=1)/80_000)
lo, hi = diff - 1.96*se, diff + 1.96*se
d = diff / np.sqrt((control.var(ddof=1) + variant.var(ddof=1))/2)

print(f"difference  {diff:+.4f} seconds")
print(f"95% CI      [{lo:+.4f}, {hi:+.4f}]")
print(f"Cohen's d   {d:+.4f}   (0.2 is considered SMALL)")
print(f"p-value     {stats.ttest_ind(control, variant, equal_var=False).pvalue:.2e}")
print()
print("Honest version: 'Time on page rose by 0.03s (95% CI 0.00 to 0.06).")
print("That is far below the 0.5s we agreed would justify shipping.'")

### B4 — the null result

A study tests whether a supplement affects reaction time, and reports no effect.

In [ ]:
rng_b4 = np.random.default_rng(404)
placebo = rng_b4.normal(250, 40, 9)      # milliseconds
treated = rng_b4.normal(235, 40, 9)      # a real 15ms improvement

res = stats.ttest_ind(placebo, treated, equal_var=False)
print(f"p = {res.pvalue:.3f}  ->  no significant effect.")
print("We conclude the supplement does not affect reaction time.")

**Bug:** the conclusion. "We did not find an effect" has been turned into "there is no effect",
and with n = 9 per group the study had almost no chance of finding one.

**What the wrong conclusion costs:** there *is* a real 15 ms improvement here. A two-sample
test with 9 per group and d ≈ 0.375 has roughly **12%** power — it would miss this effect about
seven times in eight.

**Fix:** report the confidence interval, which makes the ambiguity visible, and state the power.

The interval here runs from −12 to +51 ms. That is consistent with a substantial
improvement, with nothing, and with the supplement making things worse. The honest summary is
*"this study was too small to tell"*, not *"the supplement does not work"*.

**The general rule:** a wide interval around zero means you learned nothing. A narrow interval
around zero means you learned something real — that any effect is small. Same p-value,
completely different findings.

In [ ]:
rng_b4 = np.random.default_rng(404)
placebo = rng_b4.normal(250, 40, 9)
treated = rng_b4.normal(235, 40, 9)

diff = placebo.mean() - treated.mean()
se = np.sqrt(placebo.var(ddof=1)/9 + treated.var(ddof=1)/9)
res = stats.ttest_ind(placebo, treated, equal_var=False)
lo, hi = stats.t.interval(0.95, res.df, diff, se)

print(f"observed improvement {diff:+.1f} ms")
print(f"95% CI               [{lo:+.1f}, {hi:+.1f}] ms")
print(f"p                    {res.pvalue:.3f}")
print()
print("The interval is consistent with a big improvement, with nothing,")
print("and with the supplement making things worse.")
print()

#  what power did this study actually have?
def power_analytic(n, d, alpha=0.05):
    crit = stats.norm.ppf(1 - alpha/2); ncp = d*np.sqrt(n/2)
    return stats.norm.cdf(ncp - crit) + stats.norm.cdf(-ncp - crit)

d_true = 15/40
print(f"the real effect is d = {d_true:.3f}")
print(f"{'n/group':>9} {'power':>8}")
for n in [9, 30, 60, 112, 200]:
    print(f"{n:>9} {power_analytic(n, d_true):>8.1%}")
print()
print("This study had about a 1-in-8 chance of finding an effect that was really there.")

---
# Part C — Write the code

Write the code yourself. Then say the answer in plain English.

### C1 — run a test properly, end to end

A bakery changes its recipe. 40 customers rate the old loaf and 40 different customers rate the
new one, out of 10.

1. Simulate plausible data where the new recipe really is 0.6 points better, with sd 1.8.
2. Run the correct test and say why you chose it.
3. Report the effect size, the confidence interval and the exact p-value.
4. Write the one-sentence conclusion you would give the bakery.

In [ ]:
rng_c1 = np.random.default_rng(304)

old = rng_c1.normal(6.8, 1.8, 40)
new = rng_c1.normal(7.4, 1.8, 40)        # different customers -> NOT paired

#  2. Welch's t: two independent groups, and we should not assume equal variance
res = stats.ttest_ind(old, new, equal_var=False)

#  3. the numbers that matter
diff = new.mean() - old.mean()
se   = np.sqrt(old.var(ddof=1)/40 + new.var(ddof=1)/40)
lo, hi = stats.t.interval(0.95, res.df, diff, se)
d = diff / np.sqrt((old.var(ddof=1) + new.var(ddof=1)) / 2)

print(f"old  mean {old.mean():.2f}  sd {old.std(ddof=1):.2f}  n 40")
print(f"new  mean {new.mean():.2f}  sd {new.std(ddof=1):.2f}  n 40")
print()
print(f"difference  {diff:+.3f} points")
print(f"95% CI      [{lo:+.3f}, {hi:+.3f}]")
print(f"Cohen's d   {d:+.3f}")
print(f"Welch t     {res.statistic:+.3f}  df {res.df:.1f}  p = {res.pvalue:.4f}")

**2. Why Welch's t:** these are two *different* sets of customers, so the data is not paired —
`ttest_rel` would be wrong. And we have no reason to assume the two groups have identical
variance, so `equal_var=False`.

**4. The sentence for the bakery:**

> *"The new recipe scored 1.0 points higher out of 10 (95% CI 0.15 to 1.89; d = 0.52; Welch's
> t-test, 40 customers per recipe, p = 0.022). This was the only comparison we ran."*

Note what that sentence does that "p < 0.05" does not: it gives the bakery a number they can
act on, an honest range around it — and that range is wide, running from "barely noticeable"
to "a full two points". A reader can see the imprecision, which "significant" would have
hidden.

**One thing to watch in the output:** `ttest_ind(old, new)` returns a *negative* t statistic,
because it computes old − new. The sign of `res.statistic` depends on argument order; the sign
of your reported difference should depend on what you actually want to say. Compute the
difference yourself rather than reading it off the test.

### C2 — size the study before you run it

Before collecting any data, work out how large the bakery's taste test should have been.

1. Write a `power(n, d)` function for a two-sample test.
2. Find the n per group needed for 80% power at d = 0.2, 0.33 and 0.5.
3. Plot the power curve for d = 0.33.
4. The bakery can recruit 40 per group. What is the smallest effect they can reliably detect,
   and what should you tell them?

In [ ]:
def power(n, d, alpha=0.05):
    crit = stats.norm.ppf(1 - alpha/2)
    ncp  = d * np.sqrt(n/2)
    return stats.norm.cdf(ncp - crit) + stats.norm.cdf(-ncp - crit)

print(f"{'effect d':>10} {'n per group for 80%':>22}")
for d in [0.2, 0.33, 0.5, 0.8]:
    n = next(n for n in range(4, 20_000) if power(n, d) >= 0.80)
    print(f"{d:>10} {n:>22}")

#  3. the power curve
g = np.arange(4, 400)
fig, ax = plt.subplots(figsize=(7.5, 3))
ax.plot(g, [power(n, 0.33) for n in g], color="#3E7CB1", lw=2.4)
ax.axhline(0.8, color="#152238", ls="--", lw=1.8)
ax.axvline(40, color="#E8674C", ls=":", lw=2)
ax.text(44, 0.15, "what the\nbakery has", color="#E8674C", fontsize=10)
ax.set_xlabel("n per group"); ax.set_ylabel("power"); ax.set_ylim(0, 1.03)
ax.set_title("d = 0.33")
plt.show()

#  4. what can 40 per group actually see?
d_detectable = next(d/100 for d in range(1, 300) if power(40, d/100) >= 0.80)
print(f"\n4.  at n = 40 per group, power is {power(40, 0.33):.0%} for d = 0.33")
print(f"    the smallest effect detectable with 80% power is d = {d_detectable:.2f}")
print(f"    which on a 1.8-point sd is {d_detectable*1.8:.2f} points out of 10")

**2.** 393 per group at d = 0.2, about 145 at d = 0.33, 63 at d = 0.5, 25 at d = 0.8.

**4.** With 40 per group the power at d = 0.33 is only about **31%**. The smallest effect they
can reliably detect is around **d = 0.63**, which on a 1.8-point standard deviation means a
difference of roughly **1.1 points out of 10**.

**What to tell them:**

> *"With 40 tasters per recipe you will only reliably detect a difference of about one full
> point out of ten. If the improvement you care about is half a point — d = 0.28 on this
> scale — you need around 200 per group. If you can only get 40, run the test — but understand that a null result will not mean
> the recipe is no better, only that you could not tell."*

That last clause is the whole value of doing a power calculation beforehand: it tells you in
advance what a null result will and will not be worth.

### C3 — the cost of looking

Demonstrate for yourself what peeking at the data does.

1. Simulate 4,000 experiments with **no real effect**, n = 100 per group.
2. Compute the false positive rate for one honest test at the end.
3. Now compute it for an analyst who tests after every 10 subjects from n = 20 and stops as
   soon as p < 0.05.
4. How much worse is it? And what if they peek after every single subject?

In [ ]:
rng_c3 = np.random.default_rng(283)
T = 4000

#  2. one honest test
x = rng_c3.normal(0, 1, (T, 100)); y = rng_c3.normal(0, 1, (T, 100))
honest = (stats.ttest_ind(x, y, axis=1).pvalue < 0.05).mean()

#  3 & 4. peeking at different frequencies
def peek_rate(step):
    x = rng_c3.normal(0, 1, (T, 100)); y = rng_c3.normal(0, 1, (T, 100))
    fired = np.zeros(T, bool)
    for m in range(20, 101, step):
        fired |= stats.ttest_ind(x[:, :m], y[:, :m], axis=1).pvalue < 0.05
    return fired.mean()

print(f"one honest test at n=100:        {honest:.1%}")
for step in [25, 10, 5, 1]:
    looks = len(range(20, 101, step))
    print(f"peeking every {step:>2} ({looks:>2} looks): {peek_rate(step):>6.1%}")

The false positive rate climbs steadily with the number of looks: about **5%** for one honest
test, **12.5%** peeking every 25, **16.5%** every 10, **20%** every 5, and **23%** peeking after
every single subject.

**Why:** every extra look is another chance for noise to wander across the threshold. You are
running many tests — you are simply not counting them.

The sentence that causes this is entirely reasonable-sounding: *"let's check whether it's
significant yet before we spend more budget."* Nobody saying it thinks they are cheating.

**What to do instead:** fix n in advance, or use a design built for interim looks (a sequential
test with spending functions), which adjusts the threshold for exactly this reason. What you
cannot do is peek with an unadjusted threshold and report 0.05.

### C4 — correct for multiplicity

You test 20 metrics in an A/B test. Only one has a real effect.

1. Simulate it: 19 null metrics and 1 with a real effect of d = 0.5, n = 60 per group.
2. How many come back significant at the raw 0.05 threshold?
3. Apply Bonferroni (0.05/20) and Benjamini–Hochberg. How many survive each?
4. Run the whole thing 2,000 times. Which correction would you recommend, and why?

In [ ]:
rng_c4 = np.random.default_rng(284)
K, n_eff, n = 20, 1, 60

def one_run():
    ps = []
    for k in range(K):
        d = 0.5 if k == 0 else 0.0            # metric 0 is the real one
        a = rng_c4.normal(0, 1, n)
        b = rng_c4.normal(d, 1, n)
        ps.append(stats.ttest_ind(a, b, equal_var=False).pvalue)
    return np.array(ps)

ps = one_run()
print(f"raw p < 0.05:        {(ps < 0.05).sum()} significant  "
      f"(metric 0 included: {ps[0] < 0.05})")
print(f"Bonferroni p<0.0025: {(ps < 0.05/K).sum()} significant  "
      f"(metric 0 included: {ps[0] < 0.05/K})")

#  Benjamini-Hochberg
order = np.argsort(ps)
bh = np.zeros(K, bool)
for rank, idx in enumerate(order, 1):
    if ps[idx] <= rank/K * 0.05:
        bh[order[:rank]] = True
print(f"Benjamini-Hochberg:  {bh.sum()} significant  (metric 0 included: {bh[0]})")

#  4. over many runs
T = 2000
raw_fp = bon_fp = bh_fp = 0
raw_tp = bon_tp = bh_tp = 0
for _ in range(T):
    ps = one_run()
    order = np.argsort(ps); bh = np.zeros(K, bool)
    for rank, idx in enumerate(order, 1):
        if ps[idx] <= rank/K * 0.05:
            bh[order[:rank]] = True
    raw_fp += (ps[1:] < 0.05).any();     raw_tp += ps[0] < 0.05
    bon_fp += (ps[1:] < 0.05/K).any();   bon_tp += ps[0] < 0.05/K
    bh_fp  += bh[1:].any();              bh_tp  += bh[0]

print(f"\n{'method':<20} {'any false positive':>20} {'found the real one':>20}")
for name, fp, tp in [("raw 0.05", raw_fp, raw_tp), ("Bonferroni", bon_fp, bon_tp),
                     ("Benjamini-Hochberg", bh_fp, bh_tp)]:
    print(f"{name:<20} {fp/T:>19.1%} {tp/T:>19.1%}")

The three methods trade off exactly as advertised:

- **Raw 0.05** finds the real effect **80%** of the time, and produces at least one false
  positive in **62%** of runs.
- **Bonferroni** cuts false positives to **5.5%** and finds the real effect only **38%** of the
  time — less than half as often.
- **Benjamini–Hochberg** gives **7%** false positives and **38%** power.

**The honest surprise:** BH and Bonferroni perform almost identically here. BH is usually
described as the more powerful correction, and it is — but only when there are *several* true
effects among the tests. With exactly one real effect out of twenty there is nothing for BH's
step-up procedure to exploit, so it collapses towards Bonferroni.

That is worth knowing before you reach for BH expecting free power.

**Which to recommend:** it depends on what a mistake costs.

- Bonferroni when a false positive is expensive and you are testing a handful of pre-specified
  things — a clinical endpoint, a go/no-go decision.
- Benjamini–Hochberg when you are screening many things and expect to follow up — genomics,
  exploratory dashboards, metric sweeps. Accepting a few false leads is fine if the next step
  is confirmation.
- **Neither is as good as deciding your primary metric before the test.** Correction is damage
  control for a design that tested twenty things; it is not a substitute for deciding which one
  mattered.

### C5 — rewrite a bad report

Here is a results paragraph of the kind that gets published:

> *"Participants in the intervention group showed significantly improved outcomes compared to
> controls (p < 0.05). We also observed improvements in secondary measures. These findings
> suggest the intervention is effective."*

1. List every question a reader cannot answer from this.
2. Simulate plausible data behind it: 45 per group, a real effect of d = 0.45, plus six
> secondary measures with no effect.
3. Compute everything a proper report needs.
4. Rewrite the paragraph honestly.

In [ ]:
rng_c5 = np.random.default_rng(286)
n = 45

control = rng_c5.normal(50, 10, n)
treated = rng_c5.normal(54.5, 10, n)           # d = 0.45
secondary = [(rng_c5.normal(50, 10, n), rng_c5.normal(50, 10, n)) for _ in range(6)]

res = stats.ttest_ind(control, treated, equal_var=False)
diff = treated.mean() - control.mean()
se = np.sqrt(control.var(ddof=1)/n + treated.var(ddof=1)/n)
lo, hi = stats.t.interval(0.95, res.df, diff, se)
d = diff / np.sqrt((control.var(ddof=1) + treated.var(ddof=1))/2)

print("PRIMARY OUTCOME")
print(f"  difference {diff:+.2f} points   95% CI [{lo:+.2f}, {hi:+.2f}]")
print(f"  d = {d:.3f}   Welch t({res.df:.0f}) = {res.statistic:.2f}   p = {res.pvalue:.4f}")

print("\nSECONDARY OUTCOMES (7 tests in total, so Bonferroni threshold = 0.0071)")
sig = 0
for i, (a, b) in enumerate(secondary, 1):
    p = stats.ttest_ind(a, b, equal_var=False).pvalue
    flag = " <- 'significant' at raw 0.05" if p < 0.05 else ""
    sig += p < 0.05
    print(f"  measure {i}: p = {p:.4f}{flag}")
print(f"\n  {sig} of 6 null measures crossed the raw 0.05 line")

def power(n_, d_, alpha=0.05):
    crit = stats.norm.ppf(1-alpha/2); ncp = d_*np.sqrt(n_/2)
    return stats.norm.cdf(ncp-crit) + stats.norm.cdf(-ncp-crit)
print(f"\n  power for d = 0.45 at n = {n}: {power(n, 0.45):.0%}")

**1. What a reader cannot answer:** How big was the effect? How precisely was it measured? How
many outcomes were tested, and was the primary one chosen in advance? What was the sample size?
Was "p < 0.05" 0.049 or 0.0001? What test was used? What is "improved outcomes" measured in?
Which secondary measures improved, and by how much?

**4. The honest rewrite:**

> *"The intervention group scored 4.6 points higher on the primary outcome than controls (95%
> CI 0.6 to 8.7; Cohen's d = 0.48; Welch's t-test, n = 45 per group, p = 0.026). The primary
> outcome was specified before data collection. We additionally examined six secondary
> measures; one reached p = 0.045 unadjusted, but this does not survive correction for the
> seven comparisons (Bonferroni threshold 0.0071) and is consistent with chance, so we do not
> interpret it as evidence of an effect. The study had approximately 57% power to detect an
> effect of d = 0.45, so our null findings should be read with that in mind."*

**What changed, and why it matters:** the effect now has a size and a unit. The interval shows
the plausible range runs from "barely anything" to "substantial". The secondary measures are
declared rather than hinted at — and the one that reached p = 0.045 is correctly described as
what you would expect from running seven tests, rather than smuggled in as "we also observed
improvements". The power statement lets a reader judge the null findings too.

Note the power line in particular: at 57%, this study would have *missed* a real d = 0.45
effect nearly half the time. Saying so is what stops a reader over-reading the six null
secondary measures.

It is longer, and it is the version a reader can actually act on.

---
# Part D — Challenge

### D1 — p-hack on purpose

This exercise asks you to cheat deliberately. Doing it yourself is the fastest way to recognise
it being done to you.

You are given a dataset with **absolutely no real effect in it**. Your job is to find a
"significant" result anyway, using only techniques that a reader of your write-up would never
see.

1. Generate two groups of 120 from the *same* distribution, plus five harmless-looking
   covariates (age group, region, device, signup month, plan tier).
2. Hunt for a significant result. Allowed moves: test subgroups, test different outcome
   definitions, drop "outliers", stop early. Track **every** test you run.
3. Report how many tests it took and what the honest Bonferroni threshold would have been.
4. Write the misleading one-line claim you *could* have published — and then the honest version.

In [ ]:
rng_d1 = np.random.default_rng(2802)
n = 120

#  1. two groups from the SAME distribution. There is nothing here.
df = pd.DataFrame({
    "group":  np.repeat(["A", "B"], n),
    "value":  rng_d1.normal(100, 15, 2*n),
    "age":    rng_d1.choice(["18-30", "31-45", "46-60", "60+"], 2*n),
    "region": rng_d1.choice(["North", "South", "East", "West"], 2*n),
    "device": rng_d1.choice(["mobile", "desktop"], 2*n),
    "month":  rng_d1.choice(["Jan", "Feb", "Mar"], 2*n),
    "tier":   rng_d1.choice(["free", "paid"], 2*n),
})
a = df[df.group == "A"].value.values
b = df[df.group == "B"].value.values
print(f"the truth: both groups drawn from Normal(100, 15). No effect whatsoever.")
print(f"overall test: p = {stats.ttest_ind(a, b, equal_var=False).pvalue:.4f}  "
      f"-> nothing. Time to get creative.")

In [ ]:
tests = []        # every single test we run goes in here

def try_test(label, x, y):
    if len(x) < 5 or len(y) < 5:
        return None
    p = stats.ttest_ind(x, y, equal_var=False).pvalue
    tests.append((label, len(x), len(y), p))
    return p

#  move 1: every subgroup of every covariate
for col in ["age", "region", "device", "month", "tier"]:
    for lvl in sorted(df[col].unique()):
        sub = df[df[col] == lvl]
        try_test(f"{col}={lvl}",
                 sub[sub.group == "A"].value.values,
                 sub[sub.group == "B"].value.values)

#  move 2: "obvious" two-way splits
for c1, c2 in [("device", "tier"), ("device", "age"), ("tier", "region")]:
    for l1 in df[c1].unique():
        for l2 in df[c2].unique():
            sub = df[(df[c1] == l1) & (df[c2] == l2)]
            try_test(f"{c1}={l1} & {c2}={l2}",
                     sub[sub.group == "A"].value.values,
                     sub[sub.group == "B"].value.values)

#  move 3: drop 'outliers'
for k in [1.5, 2.0, 2.5, 3.0]:
    m, s = df.value.mean(), df.value.std()
    sub = df[(df.value - m).abs() < k*s]
    try_test(f"outliers beyond {k} sd removed",
             sub[sub.group == "A"].value.values,
             sub[sub.group == "B"].value.values)

#  move 4: stop early
for m in range(30, 121, 10):
    try_test(f"stopped at n={m}", a[:m], b[:m])

hits = sorted([t for t in tests if t[3] < 0.05], key=lambda t: t[3])
print(f"tests run: {len(tests)}")
print(f"'significant' results found: {len(hits)}\n")
for lbl, na, nb, p in hits:
    print(f"  p = {p:.4f}   {lbl}   (n = {na} vs {nb})")

In [ ]:
#  3. the honest accounting
k = len(tests)
best = hits[0] if hits else None
print(f"tests actually run        {k}")
print(f"Bonferroni threshold      {0.05/k:.5f}")
print(f"P(>=1 false positive at raw 0.05 with {k} tests) = {1 - 0.95**k:.1%}")
print()
if best:
    print(f"best p-value found        {best[3]:.4f}")
    print(f"survives Bonferroni?      {best[3] < 0.05/k}")
print()
#  and how many 'hits' would you expect from pure noise?
print(f"expected false positives from {k} null tests: {k*0.05:.1f}")
print(f"we found: {len(hits)}")

**What happened.** 49 tests on data containing nothing produced one result below 0.05
(p = 0.015, mobile users on the free tier). With 49 tests the chance of at least one false
positive was **92%** — finding "something" was close to guaranteed before any data existed.

Worth noting honestly: the arithmetic expects about 2.5 false positives from 49 null tests and
we got 1, so this run was on the *lucky* side. Re-run the cell with a different seed and you
will typically get two or three to choose from.

**4. The claim I could have published:**

> *"Among mobile users on the free tier, group B scored significantly higher than group A
> (p = 0.015)."*

Every word of that is true. It is also entirely meaningless, and nothing in the sentence lets a
reader detect the problem. That is what makes it dangerous — it is not a lie, it is a true
statement with the context removed.

**The honest version:**

> *"We found no difference between groups overall (p = 0.36, n = 120 per group). We explored 49
> subgroup and sensitivity analyses; 1 reached p < 0.05, against the 2.5 expected by chance at
> this number of comparisons. It does not survive correction (Bonferroni threshold 0.001). We
> treat it as a hypothesis to test in a new sample, not as a finding."*

**Three things to take away:**

1. **The techniques are all individually defensible.** Subgroup analysis is legitimate. Removing
   outliers is legitimate. Stopping when you have enough data is legitimate. It is doing them
   *and then reporting only the winner* that produces the fiction.
2. **The write-up is where it happens.** Nothing in the analysis was fraudulent; the deception
   was entirely in what was omitted.
3. **This is why pre-registration exists.** Writing down the analysis before seeing the data
   removes the degrees of freedom that make this possible — not because researchers are
   dishonest, but because the flexibility is invisible to everyone, including yourself.

### D2 — audit a published claim

A company press release reports:

> *"Our new onboarding flow increased 30-day retention from 22% to 26% — an 18% relative
> improvement — in a test with 400 users per variant (p = 0.04)."*

1. Reconstruct the test. Is p = 0.04 consistent with those numbers?
2. Compute the confidence interval for the difference. What range of true effects is compatible
   with this result?
3. What power did the test have to detect the claimed effect? And what would a *replication*
   with the same n find?
4. The "18% relative improvement" is doing a lot of work. Explain why, and say what you would
   send back.

In [ ]:
#  1. reconstruct it: a two-proportion test
p1, p2, n = 0.22, 0.26, 400
x1, x2 = int(p1*n), int(p2*n)

pooled = (x1 + x2) / (2*n)
se_pool = np.sqrt(pooled*(1-pooled)*(2/n))
z = (p2 - p1) / se_pool
p_val = 2*(1 - stats.norm.cdf(abs(z)))

print(f"retained: {x1}/400 = {p1:.0%}   vs   {x2}/400 = {p2:.0%}")
print(f"z = {z:.3f}   p = {p_val:.4f}")
print(f"the claimed p = 0.04 is {'consistent' if 0.03 < p_val < 0.12 else 'NOT consistent'}"
      f" with these numbers (we get {p_val:.3f})")

In [ ]:
#  2. the confidence interval for the difference
se_diff = np.sqrt(p1*(1-p1)/n + p2*(1-p2)/n)
lo, hi = (p2-p1) - 1.96*se_diff, (p2-p1) + 1.96*se_diff
print(f"absolute difference {p2-p1:+.1%}")
print(f"95% CI              [{lo:+.1%}, {hi:+.1%}]")
print()
print(f"in relative terms, the CI runs from {lo/p1:+.0%} to {hi/p1:+.0%}")
print("so the data is compatible with anything from 'no better at all'")
print("to 'more than a third better'.")

In [ ]:
#  3. power, and what a replication would find
h = 2*np.arcsin(np.sqrt(p2)) - 2*np.arcsin(np.sqrt(p1))      # Cohen's h
def power_prop(n_, h_, alpha=0.05):
    crit = stats.norm.ppf(1-alpha/2); ncp = abs(h_)*np.sqrt(n_/2)
    return stats.norm.cdf(ncp-crit) + stats.norm.cdf(-ncp-crit)

print(f"Cohen's h for 22% -> 26%:  {h:.4f}  (0.2 is 'small')")
print(f"power at n = 400/arm:      {power_prop(400, h):.0%}")
print()
print(f"{'n per arm':>11} {'power':>8}")
for nn in [400, 800, 1600, 2500]:
    print(f"{nn:>11} {power_prop(nn, h):>8.0%}")
print()
#  simulate replications
rng_d2 = np.random.default_rng(2803)
T = 20_000
r1 = rng_d2.binomial(400, 0.22, T)/400
r2 = rng_d2.binomial(400, 0.26, T)/400
pooled_ = (r1+r2)/2
se_ = np.sqrt(pooled_*(1-pooled_)*(2/400))
reps = (np.abs(r2-r1)/se_ > 1.96)
print(f"if the effect is exactly as claimed, a replication at the same n")
print(f"would reach p < 0.05 only {reps.mean():.0%} of the time")

**1.** The numbers are consistent — a two-proportion test on 88/400 versus 104/400 gives
z = 1.33, p = 0.185. Note that this does **not** match the claimed p = 0.04, which is the first
thing worth flagging: either the release rounded the proportions, used a one-sided test, or the
p-value came from somewhere else. (A one-sided test gives 0.093, still not 0.04.)

**2.** The 95% CI for the absolute difference runs from **−1.9 to +9.9 percentage
points**. In relative terms that is **−9% to +45%**. The data is compatible with the new flow
being slightly *worse*, and with it being far better than claimed.

**3.** Cohen's h here is 0.094 — well under "small". At 400 per arm the test had roughly
**26%** power. A replication with the same sample size, assuming the effect is exactly as
claimed, would reach significance only about a quarter of the time. You would need around
**1,800 per arm** for 80% power.

**4. Why "18% relative" is doing the work.** The absolute difference is 4 percentage points.
Dividing by a small baseline turns that into an impressive-sounding 18%. Relative figures on
small baselines always look larger, and they hide the precision: "18% improvement" sounds firm,
while "somewhere between slightly worse and a third better" is the same information stated
honestly.

**What I would send back:**

> *"We cannot reproduce p = 0.04 from the figures given — 88/400 versus 104/400 gives p = 0.185
> two-sided, or 0.093 one-sided. Could you confirm which test was used? Separately, the 95%
> confidence interval for the difference runs from −1.9 to +9.9 percentage points, so the
> result is also compatible with no improvement at all. The study had about 26% power to detect
> the effect you are reporting, which means a result this size is as likely to be noise as
> signal — a replication at the same sample size would reach significance only about a quarter
> of the time. We would want roughly 1,800 users per variant, and we would suggest leading with
> the absolute difference in percentage points rather than the relative figure."*

**The general lesson:** a relative improvement on a small base, a borderline p-value and an
underpowered study are the three ingredients of almost every overstated result you will read.
All three are visible here from numbers printed in the release itself.

---
# Where the marks were

| Question | The thing being tested |
|---|---|
| A1 | p-values are uniform under the null |
| A2 | ⚠️ scipy's default is Student's t, and it can fire 36% |
| A3 | multiplicity arithmetic, both directions |
| A4 | ⚠️ paired data needs a paired test — 0% vs 98% |
| A5 | significant means detectable, not important |
| A6 | the base rate decides what p < 0.05 is worth |
| A7 | why t and not z, and when it stops mattering |
| A8 | a test only sees the gap in standard errors |
| B1 | the default that invents an effect |
| B2 | the wrong test that discards a real one |
| B3 | a correct test with a wrong conclusion |
| B4 | "not significant" from an underpowered study |
| C1 | running and reporting a test properly |
| C2 | sizing a study, and what a null result will be worth |
| C3 | the cost of peeking, measured |
| C4 | Bonferroni vs Benjamini–Hochberg, and their trade-off |
| C5 | rewriting a bad report into an honest one |
| D1 | p-hacking from the inside |
| D2 | auditing a claim from its own published numbers |

If you only remember three things from today:

1. **A p-value is P(data \| null).** Not the chance the null is true, not the chance you are
   right, and not a measure of how big the effect is.
2. **Two one-word fixes:** `equal_var=False`, and `ttest_rel` when the data is paired. Both
   change the answer completely and neither raises an error.
3. **Report the effect, the interval, and how many tests you ran.** The verdict is the least
   informative thing you can publish.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*